# Spaceship Titanic - XGBoost Baseline

Simple baseline: median fill numerics, label encode categoricals, XGBoost with 5-fold CV.

**Expected CV**: ~0.80 | **LB**: 0.799

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Target balance: {train['Transported'].mean():.3f}")

## Feature Engineering (minimal)

- Median fill for numeric columns
- Mode fill for categoricals
- Label encode categoricals
- Map booleans to 0/1

In [ ]:
# Combine train/test for consistent encoding
train['is_train'] = 1
test['is_train'] = 0
test['Transported'] = np.nan
combined = pd.concat([train, test], axis=0, ignore_index=True)

# Numeric: median fill
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    combined[col] = combined[col].fillna(combined[col].median())
combined['Age'] = combined['Age'].fillna(combined['Age'].median())

# Categorical: mode fill
combined['HomePlanet'] = combined['HomePlanet'].fillna(combined['HomePlanet'].mode()[0])
combined['Destination'] = combined['Destination'].fillna(combined['Destination'].mode()[0])

# Boolean: map to 0/1
combined['CryoSleep'] = combined['CryoSleep'].map({True: 1, False: 0, 'True': 1, 'False': 0})
combined['CryoSleep'] = combined['CryoSleep'].fillna(combined['CryoSleep'].mode()[0])
combined['VIP'] = combined['VIP'].map({True: 1, False: 0, 'True': 1, 'False': 0})
combined['VIP'] = combined['VIP'].fillna(0)

# Label encode categoricals
for col in ['HomePlanet', 'Destination']:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# Split back
train_df = combined[combined['is_train'] == 1].copy()
test_df = combined[combined['is_train'] == 0].copy()

y = train_df['Transported'].astype(int)
feature_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP'] + spend_cols
X = train_df[feature_cols].values
X_test = test_df[feature_cols].values
print(f"Features ({len(feature_cols)}): {feature_cols}")

## XGBoost 5-Fold CV